In [1]:
using Revise
using InteractiveUtils

includet("phase0/functions/load_phase0.jl")


QoG Time-Series Loader Pipeline
Input: data/qog_std_ts_jan25.arrow

>>> Step 1/5: Loading raw data with identity promotion...
    Loaded: 12391 rows × 2010 columns
    ✓ ggis_rowid assigned

>>> Step 2/5: Previewing rescue collisions...
>>> RESCUE COLLISION PREVIEW (informational — NO rows will be deleted):
    Total (ccode, year) pairs with >1 row: 22

    By entity combination:
      VDR + VNM: 22 years (1955-1976)
        Years: 1955, 1956, 1957, 1958, 1959, 1960, 1961, 1962, 1963, 1964, 1965, 1966, 1967, 1968, 1969, 1970, 1971, 1972, 1973, 1974, 1975, 1976

    NOTE: Use `ggis_rowid` as unique key, or (ident_ccode, ident_year, ident_ccodealp)
    ⚠️  Found 1 collision group(s)
    (See year details above)

>>> Step 3/5: Rescuing historical ccodes...
>>> Historical Ccode Rescue:
    Missing before: 234
    Rescued: 234
    Missing after: 0
    Row count: 12391 (unchanged)
    By alpha code:
      ETH → 231: 47 rows
      YEM → 887: 44 rows
      DEU → 276: 42 rows
      MHL → 584: 3

In [2]:

dataframe_summaries()

=== DataFrame: REGION_LABELS ===
10×2 DataFrame

=== DataFrame: df ===
12391×2013 DataFrame

=== DataFrame: meta_df ===
2010×8 DataFrame



In [3]:
meta_plus1 = enrich_metadata_with_lifespan();


>>> Results Summary
    Variables Audited: 2009
      :modern — 887
      :experimental — 355
      :legacy — 240
      :current — 240
      :historical — 168
      :anchor — 119


In [4]:
check_year_discrepancies(meta_plus1)

No year discrepancies found exceeding tolerance 1.


In [5]:

meta = meta_plus1
# Pre-compute the universe as the script does
regional_universe = compute_regional_country_universe(df);

In [6]:
"""
    inspect_slug_geo(df, meta_df, slug, regional_universe)

Inspects the geographic data availability for a specific slug at its 
recorded death year. 'slug' can be a String or a Symbol.
"""
function inspect_slug_geo(df, meta_df, slug, regional_universe)
    # Convert slug to both formats for consistency
    slug_text = string(slug)
    slug_sym = Symbol(slug)

    # 1. Identify the target year being used for this slug
    target_row = subset(meta_df, :slug => ByRow(isequal(slug_text)))
    
    if nrow(target_row) == 0
        println("❌ Error: Slug '$slug_text' not found in metadata.")
        return
    end

    target_year = target_row.max_year[1]
    
    if ismissing(target_year)
        println("❌ Error: ggis_death_year is missing for '$slug_text'.")
        return
    end

    println("\n" * "="^40)
    println("INSPECTION: $slug_text")
    println("="^40)
    println("Target (Death) Year: ", target_year)

    # 2. Check data availability in main timeseries
    year_mask = df.ident_year .== target_year
    # Ensure we don't error if slug_sym doesn't exist in df
    if !(slug_sym in propertynames(df))
        println("❌ Error: Column :$slug_sym not found in main DataFrame.")
        return
    end
    
    slug_mask = .!ismissing.(df[!, slug_sym])
    valid_rows = df[year_mask .& slug_mask, :]
    
    println("Rows found with data: ", nrow(valid_rows))

    # 3. Regional distribution and Denominators
    if nrow(valid_rows) > 0
        # Distribution in the data
        counts = combine(groupby(valid_rows, :ggis_region), nrow => :count)
        println("\n--- Regional Distribution (Numerators) ---")
        println(counts)

        # Matching denominators from the universe
        found_regions = counts.ggis_region
        denoms = filter(r -> r.ident_year == target_year && r.ggis_region in found_regions, regional_universe)
        
        println("\n--- Regional Universe (Denominators) for $target_year ---")
        println(denoms)
        
        # Calculate penetration on the fly for verification
        println("\n--- Calculated Penetrations ---")
        for row in eachrow(counts)
            d_row = filter(r -> r.ggis_region == row.ggis_region, denoms)
            if !isempty(d_row)
                d = d_row.total_countries_in_region[1]
                p = round(row.count / d, digits=3)
                println("Region $(row.ggis_region): $(row.count) / $d = $p")
            end
        end
    else
        println("⚠️ ALERT: No data found for this slug in its recorded death year.")
    end
    println("="^40 * "\n")
end

inspect_slug_geo

In [7]:
# # 1. Filter for regional variables and select relevant columns
# regional_vars = subset(meta_plus2, :ggis_geo_classification => ByRow(isequal(:regional)))

# # 2. Get unique combinations of prefix and the penetration vector
# # We use 'unique' on the selected columns to see the distinct "footprints" per prefix
# unique_regional_profiles = unique(regional_vars[!, [:prefix, :ggis_region_penetration]])

# # 3. Sort for better readability
# sort!(unique_regional_profiles, :prefix)

# # 4. Display the results
# display(unique_regional_profiles)

In [11]:


regional_vars = subset(meta_plus2,
    :ggis_geo_classification => ByRow(isequal(:regional)),
    skipmissing=true
)

regional_summary = combine(groupby(regional_vars, :prefix)) do sdf
    standard = mode(sdf.ggis_region_penetration)  # scalar (or possibly a vector object)

    return (
        Regional_Footprint = Ref(standard),
        Slugs_Shown = nrow(sdf)
    )
end

sort!(regional_summary, :Slugs_Shown, rev=true)
display(regional_summary)


Row,prefix,Regional_Footprint,Slugs_Shown
,String7,RefValue…,Int64
1,aii,"RefValue{Vector{Float64}}([0.0, 0.0, 0.25, 0.979592, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0])",64
2,oecd,"RefValue{Vector{Float64}}([0.321429, 0.15, 0.1, 0.0204082, 0.814815, 0.333333, 0.0, 0.0, 0.0, 0.0])",59
3,cpds,"RefValue{Vector{Float64}}([0.407407, 0.0, 0.05, 0.0, 0.851852, 0.166667, 0.0, 0.0, 0.0, 0.0])",49
4,sgi,"RefValue{Vector{Float64}}([0.392857, 0.1, 0.15, 0.0, 0.851852, 0.333333, 0.0, 0.0, 0.0, 0.0])",29
5,iiag,"RefValue{Vector{Float64}}([0.0, 0.0, 0.25, 0.979592, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0])",21
6,eu,"RefValue{Vector{Float64}}([0.821429, 0.0, 0.1, 0.0, 0.814815, 0.0, 0.0, 0.0, 0.0, 0.0])",12


In [19]:

names(meta_df)

8-element Vector{String}:
 "slug"
 "prefix"
 "label"
 "description"
 "type"
 "provenance"
 "min_year"
 "max_year"

In [9]:
slug = "aii_aio"
inspect_slug_geo(df, meta_df, slug, regional_universe)


INSPECTION: aii_aio
Target (Death) Year: 2022
Rows found with data: 53

--- Regional Distribution (Numerators) ---
2×2 DataFrame
 Row │ ggis_region  count 
     │ Int64?       Int64 
─────┼────────────────────
   1 │           3      5
   2 │           4     48

--- Regional Universe (Denominators) for 2022 ---
2×3 DataFrame
 Row │ ident_year  ggis_region  total_countries_in_region 
     │ Int64       Int64?       Int64                     
─────┼────────────────────────────────────────────────────
   1 │       2022            3                         20
   2 │       2022            4                         49

--- Calculated Penetrations ---
Region 3: 5 / 20 = 0.25
Region 4: 48 / 49 = 0.98



In [22]:

slug = "iiag_he"
inspect_slug_geo(df, meta_df, slug, regional_universe)


INSPECTION: iiag_he
Target (Death) Year: 2021
Rows found with data: 53

--- Regional Distribution (Numerators) ---
2×2 DataFrame
 Row │ ggis_region  count 
     │ Int64?       Int64 
─────┼────────────────────
   1 │           3      5
   2 │           4     48

--- Regional Universe (Denominators) for 2021 ---
2×3 DataFrame
 Row │ ident_year  ggis_region  total_countries_in_region 
     │ Int64       Int64?       Int64                     
─────┼────────────────────────────────────────────────────
   1 │       2021            3                         20
   2 │       2021            4                         49

--- Calculated Penetrations ---
Region 3: 5 / 20 = 0.25
Region 4: 48 / 49 = 0.98



In [10]:
meta_plus2 = enrich_metadata_with_geographic_coverage(meta_plus1);


>>> Computing Geographic Coverage (Step 8)
    Method: Peak Year Analysis
    Global Threshold:   ≥ 0.95
    Regional Threshold: ≥ 0.8
    Pre-computing geographic universes...

>>> Geographic Classification Summary (Peak-Based)
    :other — 1038
    :global — 737
    :regional — 234


In [23]:
CSV.write("data/qog_metadata_plus2.csv", meta_plus2)

"data/qog_metadata_plus2.csv"

In [10]:
# is_global(x) =
#     !ismissing(x) &&
#     lowercase(strip(string(x))) == "global"

# starts_wdi_gdp(x) =
#     !ismissing(x) &&
#     startswith(string(x), "wdi_gdp")

# df2 = meta_plus2 |>
#     x -> subset(
#         x,
#         :slug => ByRow(starts_wdi_gdp),
#         :ggis_geo_classification => ByRow(is_global)
#     ) |>
#     x -> select(
#         x,
#         Not([:prefix, :description, :type, :provenance, :min_year, :max_year])
#     )


In [11]:
# select(
#     subset(
#         meta_plus2,
#         :slug => ByRow(s -> startswith(s, "wdi_gdp")),   # allow missing-handling by skipmissing
#         :ggis_geo_classification => ByRow(==("global"));
#         skipmissing=true
#     ),
#     Not([:label, :prefix, :description, :type, :provenance, :min_year, :max_year])
# )

In [12]:
# select(
#     subset(
#         meta_plus2,
#         :slug => ByRow(s -> !ismissing(s) && startswith(s, "wdi_gdp")),
#         :ggis_geo_classification => ByRow(g -> !ismissing(g) && g == "global"),
#     ),
#     Not([:label, :prefix, :description, :type, :provenance, :min_year, :max_year])
# )

In [13]:
# select(
#     filter(:slug => s -> startswith(s, "wdi_gdp"), meta_plus2),
#     Not(:label, :prefix, :description, :type, :provenance, :min_year, :max_year)
# )

In [6]:
run_enrich_metadata_samples();


  enrich_metadata.jl — Function intent and usage

┌─ load_dataframes()
│  INTENT: Load the main QoG timeseries and the joined metadata in one call.
│  USE WHEN: You need both df and meta_df for auditing or enrichment.
│
│  RETURNS: (df, meta_df)
│    - df: main timeseries from load_qog_timeseries()
│    - meta_df: from PATH_METADATA_JOINED
│
│  USAGE:
│    df, meta = load_dataframes()
└──────────────────────────────────────────────────────────────────────────

┌─ classify_temporal_profile(birth_year, death_year; kwargs...)
│  INTENT: Classify a variable's temporal profile from its lifespan.
│  USE WHEN: You have first/last year and want :anchor, :experimental,
│           :legacy, :historical, :current, :modern, or :unclassified.
│
│  ARGUMENTS:
│    birth_year::Int, death_year::Int (positional)
│    Optional kwargs: data_start, data_end, current_year, active_lag, thresholds
│
│  RETURNS: Symbol (e.g. :anchor, :current)
│
│  USAGE:
│    profile = classify_temporal_profile(1946, 2022)
